# From Omilia Calls to a Training-Ready Dataset in Snowflake

**Welcome to Day 2 of Omilia & Snowflake Open Doors — The Art of the Possible with Cortex AI.**

In this hands-on session you will build a complete AI-powered data pipeline from raw call-center recordings to a training-ready dataset — all inside Snowflake, with zero infrastructure to manage.

Here's what you'll do:

1. **Load** raw call-center data from a shared stage
2. **Anonymize** transcripts using AI_REDACT (PII removal)
3. **Enrich** with Cortex AI (summaries, sentiment, categories)
4. **Score** with a simulated external model (Omilia-style)
5. **Transcribe** audio files (bonus)
6. **Create** a Semantic View & Cortex Agent (bonus)
7. **Generate** a dashboard with Cortex Code

---

### Before you begin
Change the `MY_SCHEMA` variable in the next cell to your assigned name (e.g., `ALEJANDRO_P`).

In [ ]:
-- ======================================================================
-- STEP 0: SETUP
-- ======================================================================

-- >> CHANGE THIS to your assigned schema name (first name + initial) <<
SET MY_SCHEMA = 'YOURNAME_X';

USE ROLE WORKSHOP_PARTICIPANT_2026;
USE WAREHOUSE WORKSHOP_WH;
USE DATABASE OMILIA_WORKSHOP_DAY2;

CREATE SCHEMA IF NOT EXISTS IDENTIFIER($MY_SCHEMA);
USE SCHEMA IDENTIFIER($MY_SCHEMA);

---
## Step 1: Load Raw Data from Shared Stage

The facilitator has pre-staged a CSV with 200 synthetic Omilia-style call records.
This represents raw production data flowing into Snowflake — the starting point of the data flywheel.

**Expected:** 200 rows loaded into `CALLS_RAW`.

In [ ]:
-- ======================================================================
-- STEP 1: Load raw call data
-- ======================================================================

CREATE OR REPLACE TABLE CALLS_RAW (
    CALL_ID VARCHAR,
    CUSTOMER_ID VARCHAR,
    CUSTOMER_NAME VARCHAR,
    CUSTOMER_EMAIL VARCHAR,
    QUEUE VARCHAR,
    LANG VARCHAR,
    CALL_START_TS TIMESTAMP_NTZ,
    CALL_END_TS TIMESTAMP_NTZ,
    DURATION_SEC INTEGER,
    AGENT_ID VARCHAR,
    CHANNEL VARCHAR,
    RAW_TRANSCRIPT TEXT
);

COPY INTO CALLS_RAW
FROM @OMILIA_WORKSHOP_DAY2.SHARED.WORKSHOP_STAGE/calls_raw.csv
FILE_FORMAT = (TYPE = 'CSV' SKIP_HEADER = 1 FIELD_OPTIONALLY_ENCLOSED_BY = '"'
               ESCAPE_UNENCLOSED_FIELD = NONE);

SELECT COUNT(*) AS rows_loaded FROM CALLS_RAW;

In [ ]:
-- Preview: notice the PII (names, emails, phone numbers) in the transcripts
SELECT CALL_ID, CUSTOMER_NAME, QUEUE, LANG, DURATION_SEC,
       LEFT(RAW_TRANSCRIPT, 120) AS transcript_preview
FROM CALLS_RAW
LIMIT 5;

---
## Step 2: Anonymize Transcripts with AI_REDACT

Before data can be used for training or analytics, PII must be removed.
Snowflake's `AI_REDACT` automatically detects and masks names, emails, phone numbers, and addresses.

> **Production note:** This step could be replaced by a call to Omilia's own anonymization model/logic
> via an external function or UDF. `AI_REDACT` is used here as a built-in, zero-setup alternative.

**Expected:** `CALLS_ANON` table with PII replaced by `[NAME]`, `[EMAIL]`, `[PHONE_NUMBER]` placeholders.

In [ ]:
-- ======================================================================
-- STEP 2: Anonymize with AI_REDACT
-- ======================================================================
-- Uses Snowflake's built-in AI_REDACT to remove PII from transcripts.
-- In production, this could be a call to your own anonymization model/logic.

CREATE OR REPLACE TABLE CALLS_ANON AS
SELECT
    CALL_ID,
    CUSTOMER_ID,
    QUEUE,
    LANG,
    CALL_START_TS,
    CALL_END_TS,
    DURATION_SEC,
    AGENT_ID,
    CHANNEL,
    AI_REDACT(RAW_TRANSCRIPT) AS TRANSCRIPT_ANON
FROM CALLS_RAW;

In [ ]:
-- Compare: raw vs anonymized transcripts
SELECT
    r.CALL_ID,
    LEFT(r.RAW_TRANSCRIPT, 100) AS original,
    LEFT(a.TRANSCRIPT_ANON, 100) AS anonymized
FROM CALLS_RAW r
JOIN CALLS_ANON a ON r.CALL_ID = a.CALL_ID
LIMIT 5;

---
## Step 3: Enrich with Cortex AI Functions

Use Snowflake Cortex AI Functions to extract structured insights from the anonymized transcripts:
- **Summary** — one-sentence call summary
- **Sentiment** — positive, negative, or neutral
- **Issue Category** — billing, technical_issue, cancellation, etc.
- **Topics** — extracted key topics
- **Escalation Flag** — should this call be escalated?

> **Timing:** This step processes 200 rows x 5 AI calls. It takes ~2-3 minutes.

In [ ]:
-- ======================================================================
-- STEP 3: Enrich with Cortex AI
-- ======================================================================

CREATE OR REPLACE TABLE CALLS_ENRICHED AS
SELECT
    *,
    TRIM(AI_COMPLETE('llama3.3-70b',
        'Summarize this call transcript in exactly one sentence. Only output the summary, nothing else.\n\nTranscript: ' || TRANSCRIPT_ANON
    ), '" \n') AS SUMMARY,

    TRIM(AI_COMPLETE('llama3.3-70b',
        'Classify the sentiment of this call as exactly one word: positive, negative, or neutral. Only output the single word.\n\nTranscript: ' || TRANSCRIPT_ANON
    ), '" \n') AS SENTIMENT,

    TRIM(AI_COMPLETE('llama3.3-70b',
        'Classify the primary issue category. Choose exactly one from: billing, technical_issue, cancellation, product_inquiry, complaint, general_inquiry. Only output the category.\n\nTranscript: ' || TRANSCRIPT_ANON
    ), '" \n') AS ISSUE_CATEGORY,

    TRIM(AI_COMPLETE('llama3.3-70b',
        'Extract the main topics from this call as a comma-separated list (max 3 topics). Only output the topics.\n\nTranscript: ' || TRANSCRIPT_ANON
    ), '" \n') AS TOPICS,

    CASE
        WHEN LOWER(TRIM(AI_COMPLETE('llama3.3-70b',
            'Should this call be escalated to a supervisor? Answer only yes or no.\n\nTranscript: ' || TRANSCRIPT_ANON
        ), '" \n')) LIKE '%yes%' THEN TRUE
        ELSE FALSE
    END AS ESCALATION_FLAG
FROM CALLS_ANON;

In [ ]:
-- Preview enriched results
SELECT CALL_ID, SUMMARY, SENTIMENT, ISSUE_CATEGORY, TOPICS, ESCALATION_FLAG
FROM CALLS_ENRICHED
LIMIT 10;

---
## Step 4: External Model Write-Back (Omilia Scoring)

In production, Omilia runs its own ML models (hosted on AWS) that score each call for:
- **Resolution probability** — likelihood the issue was resolved
- **Handling quality score** — agent performance metric
- **Escalation recommendation** — model-based escalation flag

Here we call `OMILIA_MODEL_SCORE` — a simulated version that behaves like a real external endpoint.
The results are **written back into Snowflake**, completing the data flywheel.

**Expected:** `CALLS_TRAINING_READY` — the final, fully-enriched table.

In [ ]:
-- ======================================================================
-- STEP 4: External Model Write-Back
-- ======================================================================
-- Calls the Omilia model simulation (UDF in shared schema).
-- In production: external function calling AWS/GCP endpoint.

CREATE OR REPLACE TABLE CALLS_TRAINING_READY AS
SELECT
    e.*,
    s.VALUE:resolution_probability::FLOAT AS RESOLUTION_PROBABILITY,
    s.VALUE:handling_quality_score::FLOAT AS HANDLING_QUALITY_SCORE,
    s.VALUE:escalation_recommended::BOOLEAN AS MODEL_ESCALATION_RECOMMENDED,
    s.VALUE:model_version::VARCHAR AS MODEL_VERSION
FROM CALLS_ENRICHED e,
     LATERAL (
         SELECT OMILIA_WORKSHOP_DAY2.SHARED.OMILIA_MODEL_SCORE(
             e.CALL_ID, e.TRANSCRIPT_ANON
         ) AS VALUE
     ) s;

In [ ]:
-- Final training-ready table
SELECT CALL_ID, QUEUE, SENTIMENT, ISSUE_CATEGORY,
       RESOLUTION_PROBABILITY, HANDLING_QUALITY_SCORE,
       MODEL_ESCALATION_RECOMMENDED, MODEL_VERSION
FROM CALLS_TRAINING_READY
LIMIT 10;

---
## Step 5 (Bonus): Audio Transcription

Snowflake can process audio files directly with `AI_TRANSCRIBE`.
Three sample MP3 files are pre-staged — this demonstrates audio-first data flowing into the same pipeline.

> **Note:** This step is optional. Skip to Step 6 if short on time.

In [ ]:
-- ======================================================================
-- STEP 5 (BONUS): Audio Transcription
-- ======================================================================
-- Transcribe pre-staged MP3 files into a _RAW table.

CREATE OR REPLACE TABLE AUDIO_TRANSCRIPTIONS_RAW AS
SELECT
    m.CALL_ID,
    m.AUDIO_FILE_NAME,
    AI_TRANSCRIBE(
        TO_FILE('@OMILIA_WORKSHOP_DAY2.SHARED.WORKSHOP_STAGE', m.AUDIO_FILE_NAME),
        {'timestamp_granularity': 'speaker'}
    ) AS TRANSCRIPTION_RESULT,
    CURRENT_TIMESTAMP() AS TRANSCRIBED_AT
FROM (
    SELECT 'CALL-001' AS CALL_ID, 'audiofile1.mp3' AS AUDIO_FILE_NAME
    UNION ALL SELECT 'CALL-002', 'audiofile2.mp3'
    UNION ALL SELECT 'CALL-003', 'audiofile3.mp3'
) m;

In [ ]:
-- Preview transcription results
SELECT
    CALL_ID,
    AUDIO_FILE_NAME,
    TRANSCRIPTION_RESULT:audio_duration::FLOAT AS duration_sec,
    LEFT(TRANSCRIPTION_RESULT:text::VARCHAR, 200) AS transcript_preview
FROM AUDIO_TRANSCRIPTIONS_RAW;

---
## Step 6 (Bonus): Semantic View & Cortex Agent

Create a **Semantic View** over your training-ready data — this enables natural-language querying.
Then create a **Cortex Agent** that answers questions about your call center data via text-to-SQL.

This is the foundation for building AI-powered internal tools on top of your governed data.

In [ ]:
-- ======================================================================
-- STEP 6A: Create a Semantic View
-- ======================================================================

CREATE OR REPLACE SEMANTIC VIEW CALLS_SEMANTIC_VIEW

  TABLES (
    calls AS CALLS_TRAINING_READY
      PRIMARY KEY (CALL_ID)
      COMMENT = 'Omilia call center training-ready data'
  )

  DIMENSIONS (
    calls.queue AS QUEUE
      COMMENT = 'Call queue: billing, technical_support, cancellations, sales, general',
    calls.lang AS LANG
      COMMENT = 'Language of the call: en, el, es',
    calls.channel AS CHANNEL
      COMMENT = 'Channel: phone, virtual_assistant, chat',
    calls.agent_id AS AGENT_ID
      COMMENT = 'Agent who handled the call',
    calls.call_start_ts AS CALL_START_TS
      COMMENT = 'Call start timestamp',
    calls.sentiment AS SENTIMENT
      COMMENT = 'Call sentiment: positive, negative, neutral',
    calls.issue_category AS ISSUE_CATEGORY
      COMMENT = 'Primary issue category',
    calls.escalation_flag AS ESCALATION_FLAG
      COMMENT = 'Whether the call should be escalated (TRUE/FALSE)'
  )

  METRICS (
    calls.total_calls AS COUNT(CALL_ID)
      COMMENT = 'Total number of calls',
    calls.avg_duration AS AVG(DURATION_SEC)
      COMMENT = 'Average call duration in seconds',
    calls.avg_resolution_probability AS AVG(RESOLUTION_PROBABILITY)
      COMMENT = 'Average model-predicted resolution probability (0-1)',
    calls.avg_quality_score AS AVG(HANDLING_QUALITY_SCORE)
      COMMENT = 'Average model-predicted handling quality (0-1)'
  )

  COMMENT = 'Semantic view over Omilia call training data for natural-language analytics';


In [ ]:
-- ======================================================================
-- STEP 6B: Create a Cortex Agent
-- ======================================================================

CREATE OR REPLACE AGENT CALLS_ANALYST_AGENT
  COMMENT = 'AI agent for querying Omilia call center data via natural language'
  FROM SPECIFICATION
  $$
  tools:
    - tool_spec:
        type: "cortex_analyst_text_to_sql"
        name: "CallsAnalyst"
        description: "Answers questions about call center data including volumes, sentiment, quality scores, and escalations"

  tool_resources:
    CallsAnalyst:
      semantic_view: "CALLS_SEMANTIC_VIEW"
  $$;


### Test your Agent

1. Open **Snowflake CoWork** (top-right corner of Snowsight → CoWork icon)
2. Select your **CALLS_ANALYST_AGENT** from the agent list
3. Ask it a question, for example:

```
What are the top 3 issue categories by call volume, and what is the average quality score for each?
```

The agent uses the Semantic View you just created to translate natural language into SQL and return results.

---
## Step 7: Generate Your Dashboard with Cortex Code

Open **Cortex Code (Coco)** and paste the prompt below.
Replace `<MY_SCHEMA>` with your schema name.

```
Build a polished Streamlit app in Workspaces on top of my Snowflake table
OMILIA_WORKSHOP_DAY2.<MY_SCHEMA>.CALLS_TRAINING_READY.

Columns: CALL_ID, QUEUE, LANG, DURATION_SEC, CHANNEL,
TRANSCRIPT_ANON, SUMMARY, SENTIMENT, ISSUE_CATEGORY, TOPICS,
ESCALATION_FLAG, RESOLUTION_PROBABILITY, HANDLING_QUALITY_SCORE,
MODEL_ESCALATION_RECOMMENDED.

Build:
1. KPI cards: total calls, sentiment distribution, % escalated, avg quality score
2. Filters: queue, language, escalation flag, issue category
3. A sortable result table with key columns
4. A detail panel for a selected call (summary, sentiment, topics, scores, transcript)
5. One chart: calls by queue or average quality by issue category

Constraints:
- Read from the Snowflake table directly
- Handle missing optional columns gracefully
- Simple, workshop-friendly code
- Polished Snowflake product-demo styling
```

---

### What you built today

| Layer | Table | Purpose |
|-------|-------|---------|
| Raw | `CALLS_RAW` | Production data as-is |
| Anonymized | `CALLS_ANON` | PII removed, safe for analytics |
| Enriched | `CALLS_ENRICHED` | AI-generated insights |
| Training-Ready | `CALLS_TRAINING_READY` | + External model scores |
| Audio | `AUDIO_TRANSCRIPTIONS_RAW` | Audio-first data ingestion |

This is the **data flywheel** pattern: production data flows in, gets governed and enriched,
model outputs flow back, and the whole thing powers both training and real-time analytics.

---
*Workshop created for Omilia Open Doors Day 2 | Powered by Snowflake Cortex AI*

---
## Bonus Hackathon: What Can You Build for Your Team?

Open **Cortex Code** and prompt it to build something that solves a real data challenge for your team using what you've learned today.

You can ask Cortex Code to:
- Create a new schema for your hackathon work (`YOURNAME_HACKATHON`)
- Generate mock data relevant to your team's domain
- Extract or generate insights with **Cortex AI Functions**
- Build a conversational agent with **Cortex Agents** to talk to your data
- Create a forecasting model or anomaly detection on your metrics

**This is your time to explore the future of data for your team with Snowflake.**

---

### Sample prompts to adapt

**1. Domain-specific enrichment:**
```
Using table OMILIA_WORKSHOP_DAY2.<MY_SCHEMA>.CALLS_TRAINING_READY,
add a new column called <YOUR_INSIGHT> by calling AI_COMPLETE with a prompt
that extracts <SOMETHING_YOUR_TEAM_CARES_ABOUT> from the TRANSCRIPT_ANON column.
Show me 10 sample results.
```

**2. Conversational agent for your data:**
```
Create a Cortex Agent in schema OMILIA_WORKSHOP_DAY2.<MY_SCHEMA> that can answer
natural language questions about <YOUR_TEAM'S_DATA_DOMAIN>.
First create a table with mock data representing <DESCRIBE_YOUR_DATA>,
then create a Semantic View and an Agent on top of it.
```

**3. Streamlit dashboard for your KPIs:**
```
Build a Streamlit app in Workspaces that shows <YOUR_TEAM'S_KEY_METRICS>
using data from OMILIA_WORKSHOP_DAY2.<MY_SCHEMA>.CALLS_TRAINING_READY.
Focus on <FILTERS_RELEVANT_TO_YOUR_ROLE> and highlight <WHAT_MATTERS_MOST>.
```

---
> After 15 minutes, be ready to come on stage and show what you built in 60 seconds!